# DPO Data Generation — v2

Generates a new 3k DPO training dataset with richer negative types.

**Positive side:** reuses `sft_data_ef.jsonl` — teacher-generated justifications that lead with a direct quote from the passage before connecting to the verdict. Stronger grounding than v1 positives.

**Negative taxonomy (5 types, 3000 total):**

| Type | What it targets | Source | Count |
|------|----------------|--------|-------|
| NEG1 — Wrong label | Label correctness | Reuse `dpo_neg1_3k.jsonl` | 750 |
| NEG2 — Bad reasoning | Overreach / fabrication | Reuse `dpo_neg2_3k.jsonl` | 1250 |
| NEG3 — Label hedging | Decisiveness | Constructed | 250 |
| NEG5 — Degenerate output | Repetition / garbage | Constructed | 250 |
| NEG6 — Reasoning-label mismatch | Internal consistency | Constructed | 500 |
| **Total** | | | **3000** |

NEG4 (circular citing) was dropped: GPT-4.1-mini ignored the prompt and cited the passage anyway, producing near-identical chosen/rejected pairs that contributed noise rather than signal. NEG2 is expanded to 1250 to fill the gap.

In [22]:
import os
import json
import time
import random
from collections import Counter, defaultdict
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential

os.chdir("/Users/zheng/CIS527-RL")
print("Working directory:", os.getcwd())

RESOURCE_GROUP = "cis-5270-team-10"
OPENAI_API_KEY = ""

OPENAI_ENDPOINT = f"https://{RESOURCE_GROUP}.openai.azure.com"
SUBSCRIPTION_ID = ""

os.environ["AZURE_SUBSCRIPTION_ID"] = SUBSCRIPTION_ID
os.environ["AZURE_RESOURCE_GROUP"]  = "CIS-5270"
os.environ["AZURE_AOAI_ACCOUNT"]    = RESOURCE_GROUP
os.environ["AZURE_OPENAI_API_KEY"]  = OPENAI_API_KEY
os.environ["AZURE_OPENAI_ENDPOINT"] = OPENAI_ENDPOINT

CREDENTIAL = DefaultAzureCredential()

openai_client = AzureOpenAI(
    api_key=OPENAI_API_KEY,
    azure_endpoint=OPENAI_ENDPOINT,
    api_version="2025-04-01-preview",
)

TEACHER_DEPLOYMENT = "gpt-4.1-mini"

random.seed(42)

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def save_jsonl(path, rows):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Connected to Azure OpenAI")

Working directory: /Users/zheng/CIS527-RL
Connected to Azure OpenAI


---
## Load Source Data

In [23]:
# positive side: EF-style justifications (direct passage quote + connection to verdict)
ef_data = load_jsonl("data/generated/sft_data_ef.jsonl")
print(f"EF data (chosen pool): {len(ef_data)}")
print(Counter(ex["label"] for ex in ef_data))

# neg1: wrong label + stolen justification (no API calls)
neg1_raw = load_jsonl("data/generated/dpo_neg1_3k.jsonl")
print(f"\nNEG1 pool: {len(neg1_raw)}")

# neg2: correct label + hallucinated reasoning (already generated)
neg2_raw = load_jsonl("data/generated/dpo_neg2_3k.jsonl")
print(f"NEG2 pool: {len(neg2_raw)}")

# index ef_data by id for chosen lookup
ef_by_id = {ex["id"]: ex for ex in ef_data}

EF data (chosen pool): 3000
Counter({'NOT MENTIONED': 1000, 'SUPPORTED': 1000, 'CONTRADICTED': 1000})

NEG1 pool: 1500
NEG2 pool: 1500


---
## NEG3 — Label Hedging (Constructed)

Take a correct response and wrap the label in uncertain language:
- "It could be SUPPORTED" / "possibly CONTRADICTED" / "might be NOT MENTIONED"
- Chosen is the same justification with a confident, correctly-formatted label.

Targets the model's tendency to hedge when it should commit.

In [24]:
HEDGE_TEMPLATES = [
    "It could be {label}: {justification}",
    "Possibly {label}: {justification}",
    "This might be {label}: {justification}",
    "I think {label}: {justification}",
    "Likely {label}: {justification}",
    "This seems to be {label}: {justification}",
]

neg3_pool = random.sample(ef_data, 250)
neg3_records = []
for ex in neg3_pool:
    template = random.choice(HEDGE_TEMPLATES)
    rejected = template.format(label=ex["label"], justification=ex["justification"])
    neg3_records.append({
        "id":           ex["id"],
        "neg_type":     "neg3_hedge",
        "passage":      ex["passage"],
        "claim":        ex["claim"],
        "ground_truth": ex["label"],
        "rejected_raw": rejected,
    })

print(f"NEG3 constructed: {len(neg3_records)}")
print("sample:", neg3_records[0]["rejected_raw"][:150])

NEG3 constructed: 250
sample: Possibly NOT MENTIONED: The passage only discusses the American television game show Sweethearts and its airing dates, with no mention of Helen Keller


---
## NEG4 — Dropped

NEG4 (circular citing) was removed. GPT-4.1-mini consistently ignored the instruction to avoid citing the passage and produced responses nearly identical to the chosen side, making 491/500 pairs same-label with near-zero DPO gradient. Replaced by expanding NEG2 to 1250 rows.

---
## NEG5 — Degenerate Output (Constructed)

Programmatically constructed broken responses that mimic real model failures observed in dpo_c1/c3:
- Repetition loops ("no no no no", "con con con")
- Truncated mid-sentence output
- Missing or lowercase label
- Garbled label prefix ("CONCON:", "V.V.:")

In [25]:
DEGENERATE_TEMPLATES = [
    # repetition loops — "no no no no"
    lambda lbl, just: "no " * random.randint(15, 30) + just[:40],
    # con/CONCON prefix repetition before colon (observed in dpo_c1)
    lambda lbl, just: lbl[:3].lower() + (lbl[:3].upper() + lbl[:3].lower()) * random.randint(5, 10) + ": " + just[:60],
    # truncated mid-sentence
    lambda lbl, just: f"{lbl}: " + just[:random.randint(10, 25)],
    # lowercase label
    lambda lbl, just: f"{lbl.lower()}: {just}",
    # title-case label only (e.g. "Supported: ...")
    lambda lbl, just: f"{lbl.capitalize()}: {just}",
    # label-less: starts mid-sentence with "supported by the passage..." (observed in dpo_c1)
    lambda lbl, just: f"{lbl.lower()} by the passage that {just}",
    # label-less: starts directly with justification, no label at all
    lambda lbl, just: just,
    # garbled prefix (e.g. "CONCON:", "SUPSUPSUP:")
    lambda lbl, just: (lbl[:3] * random.randint(2, 4)) + ": " + just,
]

neg5_pool = random.sample(ef_data, 250)
neg5_records = []
for ex in neg5_pool:
    template = random.choice(DEGENERATE_TEMPLATES)
    rejected = template(ex["label"], ex["justification"])
    neg5_records.append({
        "id":           ex["id"],
        "neg_type":     "neg5_degenerate",
        "passage":      ex["passage"],
        "claim":        ex["claim"],
        "ground_truth": ex["label"],
        "rejected_raw": rejected,
    })

print(f"NEG5 constructed: {len(neg5_records)}")
for r in random.sample(neg5_records, 5):
    print(" ", repr(r["rejected_raw"][:100]))

NEG5 constructed: 250
  'Contradicted: The passage states that Moscow is considered the centre of Russian culture "because of'
  'Contradicted: The passage states that "Moscow is the capital and most populous city of Russia," whic'
  'supported: The passage states that That \'70s Show "originally aired on Fox from August 23, 1998," wh'
  'NOT MENTIONED: The passage describ'
  'The passage states that Lee Harvey Oswald "was arrested for the assassination of United States Presi'


---
## NEG6 — Reasoning-Label Mismatch (Constructed)

Take a correct justification and attach the **wrong label** — the reasoning leads clearly to the right verdict but the label contradicts it.

This is the most dangerous failure mode observed in dpo_c3: the model writes a justification that correctly identifies a contradiction but then outputs SUPPORTED, or correctly grounds support but outputs CONTRADICTED. DPO directly penalizes this by contrasting identical justifications with right vs wrong labels.

In [26]:
OTHER_LABELS = {
    "SUPPORTED":     ["CONTRADICTED", "NOT MENTIONED"],
    "CONTRADICTED":  ["SUPPORTED",    "NOT MENTIONED"],
    "NOT MENTIONED": ["SUPPORTED",    "CONTRADICTED"],
}

neg6_pool = random.sample(ef_data, 500)
neg6_records = []
for ex in neg6_pool:
    wrong_label = random.choice(OTHER_LABELS[ex["label"]])
    # same justification, wrong label — the justification clearly points to the correct verdict
    rejected = f"{wrong_label}: {ex['justification']}"
    neg6_records.append({
        "id":           ex["id"],
        "neg_type":     "neg6_mismatch",
        "passage":      ex["passage"],
        "claim":        ex["claim"],
        "ground_truth": ex["label"],
        "rejected_raw": rejected,
    })

print(f"NEG6 constructed: {len(neg6_records)}")
print("sample:")
s = neg6_records[0]
ex = ef_by_id[s["id"]]
print(f"  chosen  : {ex['label']}: {ex['justification']}")
print(f"  rejected: {s['rejected_raw']}")

NEG6 constructed: 500
sample:
  chosen  : CONTRADICTED: The passage states that Joseph Stalin lived from 18 December 1878 to 5 March 1953, indicating that he died and is therefore not immortal.
  rejected: SUPPORTED: The passage states that Joseph Stalin lived from 18 December 1878 to 5 March 1953, indicating that he died and is therefore not immortal.


---
## Assemble Final DPO v2 Pairs

Combine all 5 negative types. NEG4 dropped; NEG2 expanded to 1250 rows to fill the gap.
Chosen side is always the EF-style justification from `sft_data_ef.jsonl`.
NEG1/NEG2 chosen side falls back to `sft_data_3k.jsonl` since they were generated from that pool.

In [27]:
# also need sft_data_3k for neg1/neg2 chosen fallback
sft_3k_by_id = {ex["id"]: ex for ex in load_jsonl("data/generated/sft_data_3k.jsonl")}

In [28]:
SYSTEM_PROMPT = """You are a fact-checking assistant. Given a passage and a claim, respond with the verdict followed by a one-sentence justification quoting or closely paraphrasing the passage.
Format: LABEL: justification sentence
Label must be one of: SUPPORTED, CONTRADICTED, NOT MENTIONED"""

DPO_V2_OUT = "data/generated/dpo_data_v2_3k.jsonl"

def get_rejected_text(neg_record):
    if "rejected_raw" in neg_record:
        return neg_record["rejected_raw"]
    # NEG1/NEG2 format
    return f"{neg_record['rejected_label']}: {neg_record['rejected_justification']}"

def make_pair(neg_record, chosen_ex, neg_type):
    chosen_response = f"{chosen_ex['label']}: {chosen_ex['justification']}"
    return {
        "id":       neg_record["id"],
        "neg_type": neg_type,
        "prompt":   f"Passage: {neg_record['passage']}\n\nClaim: {neg_record['claim']}",
        "chosen":   chosen_response,
        "rejected": get_rejected_text(neg_record),
    }

neg1_sample = random.sample(neg1_raw, min(750, len(neg1_raw)))
neg2_sample = random.sample(neg2_raw, min(1250, len(neg2_raw)))  # expanded from 750 to fill NEG4 gap

all_negs = [
    (neg1_sample,  sft_3k_by_id, "neg1_wrong_label"),
    (neg2_sample,  sft_3k_by_id, "neg2_bad_reasoning"),
    (neg3_records, ef_by_id,     "neg3_hedge"),
    (neg5_records, ef_by_id,     "neg5_degenerate"),
    (neg6_records, ef_by_id,     "neg6_mismatch"),
]

dpo_pairs = []
skipped = 0
for neg_list, chosen_pool, neg_type in all_negs:
    for neg in neg_list:
        chosen_ex = chosen_pool.get(neg["id"])
        if chosen_ex is None:
            skipped += 1
            continue
        dpo_pairs.append(make_pair(neg, chosen_ex, neg_type))

random.shuffle(dpo_pairs)

with open(DPO_V2_OUT, "w") as f:
    for pair in dpo_pairs:
        f.write(json.dumps(pair) + "\n")

print(f"Total DPO v2 pairs: {len(dpo_pairs)}  (skipped {skipped}) → {DPO_V2_OUT}")
print()
print("Breakdown by neg type:")
print(Counter(p["neg_type"] for p in dpo_pairs))

Total DPO v2 pairs: 3000  (skipped 0) → data/generated/dpo_data_v2_3k.jsonl

Breakdown by neg type:
Counter({'neg2_bad_reasoning': 1250, 'neg1_wrong_label': 750, 'neg6_mismatch': 500, 'neg5_degenerate': 250, 'neg3_hedge': 250})


In [29]:
# sanity check: one sample per neg type
by_type = defaultdict(list)
for p in dpo_pairs:
    by_type[p["neg_type"]].append(p)

for neg_type, pairs in sorted(by_type.items()):
    s = random.choice(pairs)
    print(f"=== {neg_type} ===")
    print(f"  chosen  : {s['chosen']}")
    print(f"  rejected: {s['rejected']}")
    print()

# confirm same-label pair rate per type
print("Same chosen==rejected label rate per type:")
def get_label(text):
    if not isinstance(text, str): return None
    t = text.upper().strip()
    for lbl in ["SUPPORTED", "CONTRADICTED", "NOT MENTIONED"]:
        if t.startswith(lbl): return lbl
    return None

for neg_type, pairs in sorted(by_type.items()):
    same = sum(1 for p in pairs if get_label(p["chosen"]) == get_label(p["rejected"]))
    print(f"  {neg_type}: {same}/{len(pairs)}")

=== neg1_wrong_label ===
  chosen  : CONTRADICTED: The passage states that Quentin Tarantino's career began in the late 1980s, not the 90s.
  rejected: SUPPORTED: The passage states that "his portrayal of Mark Bingham earned him the Boston Society of Film Critics 2006 award," indicating that Cheyenne Jackson portrayed Mark Bingham.

=== neg2_bad_reasoning ===
  chosen  : CONTRADICTED: The passage states that Peyton Manning "is the second son of former NFL quarterback Archie Manning," directly contradicting the claim.
  rejected: CONTRADICTED: Although Peyton Manning is mentioned as a son of Archie Manning, this does not necessarily mean Archie played in the NFL, as the passage could be referring to a different football league.

=== neg3_hedge ===
  chosen  : CONTRADICTED: The passage states that Roy William Whiting is "an English convicted child killer, from West Sussex," indicating he was born and associated with a location in England, which contradicts the claim that he was born and 